# Retrieval-Augmented Generation RAG

## Overview
This notebook implements a Retrieval-Augmented Generation (RAG) system specifically designed for querying company rules and regulations. The system uses LangChain framework with Groq API to provide accurate, relevant responses while filtering out irrelevant information.

## Features
- PDF document processing and chunking
- Vector embeddings for semantic search
- Groq LLM integration for intelligent responses
- Strict relevance filtering

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-groq
!pip install chromadb
!pip install pypdf
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 29.0 MB/s eta 

In [ ]:
import os
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

In [ ]:
# Used to securely store your API key
from google.colab import userdata
import os

# Load the Groq API key from Colab secrets
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

if os.environ.get('GROQ_API_KEY') is None:
    print("Error: GROQ_API_KEY not found in Colab secrets.")
else:
    print("Groq API key loaded successfully.")

Groq API key loaded successfully.


# Indexing

In [ ]:
# loading pdf

loader = PyPDFLoader("/content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf")
documents = loader.load()

In [ ]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)

In [ ]:
chunks

[Document(metadata={'producer': 'Skia/PDF m141 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'IR Solutions Curriculum & Ettiquete Rules 2025', 'source': '/content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Office  10,  3rd  floor,  Al-Rehmat  Plaza,      \n                                                                                            \nG-11\n \nMarkaz\n \nIslamabad.\n                                                                                                   hello@irsolutions.tech                                                                                                             +92  51  2761585                                                                                                              www.irsolutions.tech     \nIR  Solutions  Rules  &  Etiquette  Policy  2025'),
 Document(metadata={'producer': 'Skia/PDF m141 Google Docs Renderer', 'creator': '

# Embedding

In [ ]:
##Create Vector Store

embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k":3})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# LLM model

In [ ]:
# llm model

llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0)

# Custom Prompt

In [ ]:
# custom prompt

prompt_template = """
You are an assistant that only answers questions based on the company rules & regulations PDF.
If the answer is not found in the PDF, say: "Sorry, this information is not in the PDF."


Context from PDF:
{context}


Question:
{question}


Answer strictly based on the PDF:
"""


prompt = PromptTemplate(
input_variables=["context", "question"],
template=prompt_template
)

# Retrieval Chain

In [ ]:
## Build RetrievalQA Chain

qa_chain = RetrievalQA.from_chain_type(
llm=llm,
retriever=retriever,
chain_type="stuff",      # use for how we merge chunks
chain_type_kwargs={"prompt": prompt}      # custom instructions to control the LLM’s response
)

# Response Generator

In [ ]:
query = "What are the company SOPs give me the short summary?"
response = qa_chain.run(query)
print(response)

**Short Summary of Company SOPs (based on the PDF)**  

The SOPs section (Section 4) outlines the following key areas that every teammate must follow:

1. **SOPs & General Policies** – Standard operating procedures that govern day‑to‑day work and compliance.  
2. **Ethics** – Conduct guidelines to maintain integrity and professional behavior.  
3. **Appearance & Dress Code** – Rules for personal presentation and workplace attire.  

These SOPs are designed to preserve decorum and a positive work environment.


In [ ]:
query = "What is the company name?"
response = qa_chain.run(query)
print(response)

IR Solutions


In [ ]:
query = "Can you explain the overall overview of this pdf?"
response = qa_chain.run(query)
print(response)

**Overall Overview of the PDF**

The document is the “IR Solutions Rules & Etiquette Policy 2025.” It is a company‑wide policy manual that outlines the key rules, benefits, and procedures for employees. The main sections are:

1. **Time Management** – general guidelines on how time is tracked and managed.  
2. **Leave Management** – detailed rules for different types of leave (full day, half day, short leave, unpaid leave, marriage leave, Hajj/Umrah leave, paid/maternity leave) and the policy for annual paid leaves, including allocation, usage, and eligibility.  
3. **IR Solutions Benefits** – covers employee benefits such as JS ranks, medical claims, annual paid leaves, yearly bonus, leave encashment, remote working days, gratuity, loan policy, security cheque deposit policy, daily work progress report policy, contract renewal policy, and Slack status update policy.  
4. **SOPs & Policies** – includes standard operating procedures and general policies, ethics, appearance & dress code.

# Multi PDF RAG System

**Above system is for the one pdf. Now lets pass multiple pdf and chat with multiple pdf**

In [ ]:
# for multiple documents we have to maintain the list

pdf_paths = ["/content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf", "/content/Remote Work Policy.pdf"]  # add all your files here
documents = []

for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = path  # save filename as citation
    documents.extend(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [ ]:
prompt_template = """
You are an assistant that answers user questions.
If the answer is in the company documents, cite the document name and page number.
If the answer is not in the documents, say: "Sorry, this information is not in the provided PDFs."
If the user greets you casually (like hi, hello, what's up), respond politely without using the PDFs.

Context from documents:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

In [ ]:
# llm model

llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True  # <-- this ensures we get citations
)

In [ ]:
query = "tell me about the remote work policies?"
result = qa_chain({"query": query})

answer = result["result"]
sources = result["source_documents"]

print("Answer:", answer)
print("\nCitations:")
for doc in sources:
    print(f"- {doc.metadata['source']} (page {doc.metadata.get('page', 'N/A')})")


Answer: **Remote Work Policies (IR Solutions)**  
*Source: “Remote Work Policy” – page 3*  

1. **Eligibility & Allocation**  
   * All contract‑based teammates can use the remote‑working days defined in their contract.  
   * Annual remote days are allocated from January to December; new hires receive a prorated allocation.  
   * No remote days are allowed during probation or internship, except in emergencies (roadblocks, strikes, public holidays, etc.).  

2. **General Guidelines**  
   * Employees must complete their work responsibilities and tasks while working remotely.  
   * Regular company policies—conduct, confidentiality, code of ethics—apply during remote work.  

3. **Termination & Modification**  
   * Remote arrangements may be modified or terminated if performance standards are not met or company needs change.  
   * Management may ask employees to return to in‑office work at its discretion.  

4. **Emergency Remote Work**  
   * Employees with company laptops must brin

In [ ]:
query = "Who is eligible for the remote work?"
result = qa_chain({"query": query})

answer = result["result"]
sources = result["source_documents"]

print("Answer:", answer)
print("\nCitations:")
for doc in sources:
    print(f"- {doc.metadata['source']} (page {doc.metadata.get('page', 'N/A')})")


Answer: **Answer:**  
All teammates who are on a contract are eligible to use remote working days, as defined by the JS Ranks. Remote days are **not** available to employees who are on probation or internship.  

*Source: Remote Work Policy, page 1*

Citations:
- /content/Remote Work Policy.pdf (page 0)
- /content/Remote Work Policy.pdf (page 0)
- /content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf (page 6)


In [ ]:
query = "Is intern can also do remote work?"
result = qa_chain({"query": query})

answer = result["result"]
sources = result["source_documents"]

print("Answer:", answer)
print("\nCitations:")
for doc in sources:
    print(f"- {doc.metadata['source']} (page {doc.metadata.get('page', 'N/A')})")


Answer: Interns are not allowed to work remotely except in emergency situations (e.g., roadblocks, strikes, public holidays, or other unforeseen events).  
**Citation:** Remote Work Policy, page 1.

Citations:
- /content/Remote Work Policy.pdf (page 0)
- /content/Remote Work Policy.pdf (page 0)
- /content/Remote Work Policy.pdf (page 0)


In [ ]:
query = "Hi how are you?"
result = qa_chain({"query": query})

answer = result["result"]
sources = result["source_documents"]

print("Answer:", answer)
print("\nCitations:")
for doc in sources:
    print(f"- {doc.metadata['source']} (page {doc.metadata.get('page', 'N/A')})")


Answer: Hello! I'm doing well, thank you. How can I help you today?

Citations:
- /content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf (page 10)
- /content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf (page 0)
- /content/IR Solutions Curriculum & Ettiquete Rules 2025.pdf (page 10)


**Here i found a model is making the mistake instead of not providing the citation it is also returning the citation but the query is not relevant to any pdf so lets fix it**....

In [ ]:
def chatbot(query):
    greetings = ["hi", "hello", "hey", "what's up", "how are you", "hi there"]

    if any(greet in query.lower() for greet in greetings):
        # Pure chat response, no retrieval → no fake citations
        return "Hello 👋! I'm doing well, thank you. How can I help you today?"

    # Otherwise, run RAG
    result = qa_chain({"query": query})
    answer = result["result"]
    sources = result["source_documents"]

    if sources:
        # Extract only filename instead of full path
        citations = "\n".join([
            f"- {os.path.basename(doc.metadata['source'])} (page {doc.metadata.get('page','N/A')})"
            for doc in sources
        ])
        return f"{answer}\n\n📖 Sources:\n{citations}"
    else:
        return answer


In [ ]:
print(chatbot("Hi how are you?"))


Hello 👋! I'm doing well, thank you. How can I help you today?


In [ ]:
print(chatbot("Is anything related to time management?"))


Hello 👋! I'm doing well, thank you. How can I help you today?


In [ ]:
print(chatbot("Tell me about the company policies"))


**Company Policies (as outlined in the Remote Work Policy document, page 3)**  

1. **General Conduct & Ethics** – All employees must adhere to the company’s conduct, confidentiality, and code‑of‑ethics policies at all times, whether working in‑office or remotely.  
2. **Remote‑Work Rules** – Employees who work remotely are required to follow specific guidelines that ensure productivity, clear communication, and compliance with company standards.  
3. **Termination of Remote Work** – Remote‑work arrangements can be modified or ended if performance standards are not met or if company needs change. Management may also require employees to return to in‑office work at its discretion.  
4. **Compliance & Accountability** – Employees acknowledge that violations of any policy may lead to appropriate actions per the company’s guidelines.  

These points are summarized from the “Remote Work Policy” document (page 3).

📖 Sources:
- Remote Work Policy.pdf (page 2)
- IR Solutions Curriculum & Etti

# Use Exact Greeting Match Instead of Substring


In [ ]:
import re

def chatbot(query):
    # Normalize query
    clean_query = query.strip().lower()

    # Greeting patterns (regex ensures whole word match)
    greeting_patterns = [
        r"^hi$",
        r"^hello$",
        r"^hey$",
        r"^what's up$",
        r"^how are you$",
        r"^hi there$"
    ]

    if any(re.match(pattern, clean_query) for pattern in greeting_patterns):
        return "Hello 👋! I'm doing well, thank you. How can I help you today?"

    # Otherwise → use RAG
    result = qa_chain({"query": query})
    answer = result["result"]
    sources = result["source_documents"]

    if sources:
        citations = "\n".join([
            f"- {os.path.basename(doc.metadata['source'])} (page {doc.metadata.get('page','N/A')})"
            for doc in sources
        ])
        return f"{answer}\n\n📖 Sources:\n{citations}"
    else:
        return answer

In [ ]:
print(chatbot("Is anything related to time management?"))


Yes – the documents contain several sections that address time management.  
- **IR Solutions Rules & Etiquette Policy 2025** – Time‑management rules are outlined in sections 1.1, 1.2, and 1.3 (page 1).

📖 Sources:
- Remote Work Policy.pdf (page 2)
- IR Solutions Curriculum & Ettiquete Rules 2025.pdf (page 1)
- Remote Work Policy.pdf (page 1)
